In [2]:
import sys; sys.path.append("../src")
import pandas as pd, numpy as np, time
from common import clean_name, core_name, clean_addr, addr_numbers

s1 = pd.read_pickle("../work/tr_s1.pkl")
s2 = pd.read_pickle("../work/tr_s2.pkl")
s3 = pd.read_pickle("../work/tr_s3.pkl")
gt = pd.read_pickle("../work/tr_gt.pkl")

# ---- 1. Sample: 50k S1 + their true matches + some "no owner" records ----
s1s = s1.sample(50000, random_state=42)
g = gt[gt.matched_entity_ids != ""].copy()
g["mid"] = g.matched_entity_ids.str.split(",")
all_pairs = g[["source1_entity_id", "mid"]].explode("mid")
true_pairs = all_pairs[all_pairs.source1_entity_id.isin(s1s.entity_id)]
print("True pairs in sample:", len(true_pairs))

other = pd.concat([s2, s3])
matched_any = set(all_pairs.mid)
q_true = other[other.entity_id.isin(set(true_pairs.mid))]
q_none = other[~other.entity_id.isin(matched_any)].sample(len(q_true) // 3, random_state=42)
q = pd.concat([q_true, q_none])
del other
print("S1 records:", len(s1s), "| S2/S3 records to match:", len(q))

# ---- 2. Clean both sides ----
def prep(df):
    df = df.copy()
    df["name"] = [clean_name(x) for x in df.business_name]
    df["core"] = [core_name(x) for x in df["name"]]
    df["addr"] = [clean_addr(a, c) for a, c in zip(df.business_address, df.country)]
    df["nums"] = [addr_numbers(x) for x in df["addr"]]
    return df

t = time.time()
s1c, qc = prep(s1s), prep(q)
print("Cleaning time (s):", round(time.time() - t, 1))

# ---- 3. Build keys ----
def add_keys(df):
    w = df.core.str.split()
    df["k_first"]  = df.country + "|" + w.str[0].fillna("")
    df["k_long"]   = df.country + "|" + w.apply(lambda x: max(x, key=len) if x else "")
    df["k_num3"]   = df.country + "|" + df.nums.str.split().str[0].fillna("") + "|" + df.core.str[:3]
    df["k_sorted"] = df.country + "|" + w.apply(lambda x: " ".join(sorted(x)[:2]) if x else "")
    return df

s1c, qc = add_keys(s1c), add_keys(qc)
KEYS = ["k_first", "k_long", "k_num3", "k_sorted"]

# ---- 4. Measure each key ----
truth = set(zip(true_pairs.mid, true_pairs.source1_entity_id))
MAX_BLOCK = 200   # skip keys shared by too many S1 records

def block(key):
    a = s1c[["entity_id", key]].rename(columns={"entity_id": "s1_id"})
    a = a[~a[key].str.endswith("|")]
    sizes = a[key].value_counts()
    a = a[a[key].isin(sizes[sizes <= MAX_BLOCK].index)]
    m = qc[["entity_id", key]].merge(a, on=key)
    return set(zip(m.entity_id, m.s1_id))

union = set()
for k in KEYS:
    p = block(k)
    union |= p
    print(f"{k:10s} pairs: {len(p):>9,}   recall: {len(p & truth)/len(truth):.3f}")
print(f"{'UNION':10s} pairs: {len(union):>9,}   recall: {len(union & truth)/len(truth):.3f}")

True pairs in sample: 172636
S1 records: 50000 | S2/S3 records to match: 230181
Cleaning time (s): 9.8
k_first    pairs: 6,217,628   recall: 0.772
k_long     pairs: 6,314,411   recall: 0.618
k_num3     pairs:   305,725   recall: 0.609
k_sorted   pairs:   403,937   recall: 0.588
UNION      pairs: 11,571,062   recall: 0.876


In [3]:
found = pd.DataFrame(list(union), columns=["entity_id", "s1_id"])
tp = true_pairs.rename(columns={"source1_entity_id": "s1_id", "mid": "entity_id"})
miss = tp.merge(found, how="left", indicator=True)
miss = miss[miss["_merge"] == "left_only"].drop(columns="_merge")
miss = miss.merge(qc[["entity_id", "country", "business_name", "core", "addr"]], on="entity_id")
miss = miss.merge(s1c[["entity_id", "business_name", "core", "addr"]]
                  .rename(columns={"entity_id": "s1_id", "business_name": "s1_raw",
                                   "core": "s1_core", "addr": "s1_addr"}), on="s1_id")
print("Missed pairs:", len(miss))
print("By country:", miss.country.value_counts(normalize=True).round(3).to_dict())
for r in miss.sample(30, random_state=1).itertuples():
    print("-" * 80)
    print("S2/S3:", r.business_name, "| core:", r.core, "| addr:", r.addr)
    print("S1   :", r.s1_raw, "| core:", r.s1_core, "| addr:", r.s1_addr)

Missed pairs: 21415
By country: {'India': 0.647, 'US': 0.353}
--------------------------------------------------------------------------------
S2/S3: शिवम बिग प्रोडक्ट्स एलएलपी | core: sivm big prodkts elelpi | addr: no 2nd floor raghav plaza gill colony court road saharanpur uttr prdes
S1   : Shivam Big Products LLP | core: shivam big products | addr: uttar pradesh raghav plaza gill colony court road 2nd floor saharanpur
--------------------------------------------------------------------------------
S2/S3: scholarshipfoundation.com | core: scholarshipfoundation | addr: jarvisburg 912 michael street nc
S1   : Scholarship Foundation Corp | core: scholarship foundation | addr: 112 michael street jarvisburg nc
--------------------------------------------------------------------------------
S2/S3: pediatricspecialists.com | core: pediatricspecialists | addr: 3850 meridian avenue wichita ks
S1   : Pediatric Specialists Inc | core: pediatric specialists | addr: 3850 meridian avenue unit lot

In [1]:
import sys; sys.path.append("../src")
import pandas as pd, time, gc
from common import clean_df

for name in ["tr_s1", "tr_s2", "tr_s3", "te_s1", "te_s2", "te_s3"]:
    t = time.time()
    df = clean_df(pd.read_pickle(f"../work/{name}.pkl"))
    df.to_pickle(f"../work/clean_{name}.pkl")
    print(name, len(df), "rows,", round(time.time() - t), "s")
    del df; gc.collect()

tr_s1 2206821 rows, 40 s
tr_s2 5034616 rows, 134 s
tr_s3 5285603 rows, 177 s
te_s1 1732544 rows, 52 s
te_s2 4887273 rows, 171 s
te_s3 5082316 rows, 167 s


In [ ]:
import sys; sys.path.append("../src")
import pandas as pd, time, os
from blocking import generate_candidates_big

cols = ["entity_id", "country", "core", "addr"]
s1 = pd.read_pickle("../work/clean_te_s1.pkl")[cols]
q = pd.concat([pd.read_pickle("../work/clean_te_s2.pkl")[cols],
               pd.read_pickle("../work/clean_te_s3.pkl")[cols]], ignore_index=True)
print("S1:", len(s1), "| S2+S3:", len(q))

os.makedirs("../work/cand_test", exist_ok=True)
t = time.time()
files = generate_candidates_big(s1, q, "../work/cand_test/part")
print("DONE in", round((time.time() - t) / 60, 1), "minutes |", len(files), "files")

S1: 1732544 | S2+S3: 9969589
